# Exploratory Data Analysis (EDA)
## Smart Electrical Fault Detection System

This notebook explores `detect_dataset.csv` before we build the MLOps pipeline.

**Goal of this notebook:** understand the shape, quality, and structure of the data so that every preprocessing decision made later in `src/utils.py` and `src/train.py` is justified by evidence, not guesswork.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

df = pd.read_csv("../data/detect_dataset.csv")
df.head()

## 1. Dataset Shape

In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

**Observation:** The raw file has 12,001 rows and 9 columns. As we'll see below, 2 of those 9 columns are empty artifacts of the CSV format and will be dropped.

## 2. Column Names

In [ ]:
print(list(df.columns))

**Observation:** The columns are:
- `Output (S)` — the target label (0 = Normal, 1 = Fault)
- `Ia, Ib, Ic` — line currents for phases A, B, C
- `Va, Vb, Vc` — line voltages (per-unit) for phases A, B, C
- `Unnamed: 7`, `Unnamed: 8` — empty columns caused by two trailing commas in the raw CSV

## 3. Dataset Information

In [ ]:
df.info()

**Observation:** All 6 feature columns and the target are numeric (`int64`/`float64`), so no type conversion is needed. `Unnamed: 7` and `Unnamed: 8` have 0 non-null values — confirming they are fully empty.

## 4. Missing Values

In [ ]:
df.isnull().sum()

**Observation:** The two `Unnamed` columns are 100% missing (12,001 out of 12,001 rows). None of the real feature columns or the target have any missing values, so no imputation is required for the columns we actually use.

## 5. Duplicate Values

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")

**Observation:** There are no duplicate rows in this dataset, so no deduplication is strictly necessary — but we still include a `drop_duplicates()` step in `utils.py` as a defensive safeguard for future data updates.

## 6. Statistical Summary

In [ ]:
df.describe()

**Observation:** The current columns (`Ia, Ib, Ic`) have a much larger scale (values in the hundreds, both positive and negative) than the voltage columns (`Va, Vb, Vc`, roughly -0.6 to 0.6). This scale difference is exactly why we apply `StandardScaler` before training Logistic Regression.

## 7. Class Distribution (Target Column)

In [ ]:
counts = df["Output (S)"].value_counts()
print(counts)
print()
print(counts / len(df) * 100)

plt.figure(figsize=(5, 4))
sns.countplot(x="Output (S)", data=df)
plt.title("Class Distribution: Normal (0) vs Fault (1)")
plt.xlabel("Output (S)")
plt.ylabel("Count")
plt.show()

**Observation:** The classes are fairly balanced — about 54% Normal (0) and 46% Fault (1). This is good news: we don't need special imbalance-handling techniques like SMOTE or class weighting, which keeps the project simpler.

## 8. Correlation Matrix

In [ ]:
feature_cols = ["Ia", "Ib", "Ic", "Va", "Vb", "Vc"]
corr = df[feature_cols + ["Output (S)"]].corr()

plt.figure(figsize=(7, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Matrix")
plt.show()

**Observation:** No two features are perfectly correlated with each other (no redundant columns to drop), and several features show a meaningful correlation with the target — confirming they carry predictive signal for fault detection.

## 9. Histograms

In [ ]:
df[feature_cols].hist(figsize=(12, 8), bins=40)
plt.suptitle("Feature Distributions")
plt.tight_layout()
plt.show()

**Observation:** The current columns (`Ia, Ib, Ic`) are roughly symmetric around 0 with a wide spread, while the voltage columns (`Va, Vb, Vc`) are tightly clustered between -0.6 and 0.6. No extreme skew is present that would require a log transform.

## 10. Boxplots

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flatten(), feature_cols):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

## 11. Outlier Detection (IQR Method)

In [ ]:
def count_outliers_iqr(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

for col in feature_cols:
    n_outliers = count_outliers_iqr(df[col])
    pct = n_outliers / len(df) * 100
    print(f"{col}: {n_outliers} outliers ({pct:.2f}%)")

**Observation:** Some points flagged by the IQR rule are visible in the boxplots above. In this dataset, these "outliers" are not data errors — they are real high-magnitude readings that occur during fault events (short-circuits genuinely produce very large current spikes). Removing them would remove exactly the signal the model needs to learn, so we intentionally do **not** remove them in preprocessing.

## 12. Summary of Findings

1. The dataset has 12,001 rows and 6 usable numeric features, plus a binary target.
2. Two columns (`Unnamed: 7`, `Unnamed: 8`) are entirely empty and must be dropped.
3. There are no missing values or duplicates in the real columns.
4. The target classes are reasonably balanced (54% / 46%), so no resampling is needed.
5. Currents and voltages are on very different scales, so feature scaling is required for Logistic Regression.
6. Extreme values in the current columns are genuine fault signatures, not noise, and should be kept.

These findings directly informed the cleaning logic in `src/utils.py` and the preprocessing steps in `src/train.py`.